# Exercise 04: Index for Text Retrieval

> **Chapter:** Ch04 - Index for Text Retrieval
> **Estimated time:** ~60 minutes
>
> This exercise is optional. No submission, no grading.
> The solution notebook is released one week after the exercise.

An inverted index is what lets a search engine ignore almost the whole collection and still find
the right documents. This exercise follows that idea from cost to production. First you reason
about what a query actually reads, and why adding terms is cheap on a small catalog but expensive
on the web. Then you build a real search engine on Apache Lucene, the library behind Elasticsearch,
Solr, and OpenSearch: you index a movie collection and give it keyword search, a metadata filter,
and fuzzy matching. Finally you reason about what changes once the index no longer fits on one
machine.

**Your tasks** (tick them off as you go):

- [ ] **Quiz** - 20 questions in the quiz app
- [ ] ✏ **T1: The cost of one more term** - reason about query read cost and expansion
- [ ] 💻 **C1: A Lucene search engine** - implement indexing and querying in <a href="lucene/src/main/java/mmir/Search.java">Search.java</a>
- [ ] ✏ **T2: The same document, two scores** - explain scoring across shards

---

## Quiz

Open the quiz app and work through the **20 questions** for this chapter. On the start screen,
pick the topic **"04 - Index for Text Retrieval"**:

**Quiz app:** https://roger-weber.github.io/mmir-unibasel-hs26/quiz/

The quiz covers definitions and basic concepts. The tasks below go further: they ask you to reason
about query cost, build a working search engine yourself, and think about what breaks at scale.

---

## What a query reads

> **How this works:** Write your answer in the markdown cell below the question (replace
> *Your answer here...*). The solution notebook fills the same cell with a model answer, so you
> can compare and reflect.

### ✏ Task T1 - The cost of one more term

The chapter gives a model for how much data a query reads from an inverted index:

$$\text{entries read} \approx \frac{N \cdot K \cdot L}{M},$$

where $N$ is the number of documents, $K$ the average number of distinct terms per document, $L$ the
number of query terms, and $M$ the size of the vocabulary.

A team runs a search over 30 million e-book titles. Queries feel instant. To raise recall they turn
on query expansion: every query term is replaced by itself plus about four related terms, so a
typical three-term query becomes roughly fifteen terms. On the e-book catalog latency barely moves.
The team then deploys the same feature on a 10-billion-page web index, where the 95th-percentile
latency jumps from tens of milliseconds to several seconds.

1. Use the read-cost model to explain what expansion changes in the formula, and why the same
   fivefold expansion is almost free on the catalog but costly on the web index. Refer to the
   quantities in the formula, not just to "size".
2. To cut the cost, the team drops stop words from every expanded query. Does that bring the
   web-index latency back down? Reason about which terms dominate the read cost once the stop words
   are gone.
3. Latency is not the only price of expansion. State what expansion does to the *quality* of the
   results, and name one situation where you would enable it anyway.

> *Your answer here...*

---

## Building a search engine on Lucene

Chapter 4 ends with the practical frameworks that put an inverted index into production. Apache
Lucene is the open-source Java library at the core of most of them, including Elasticsearch,
OpenSearch, and Solr. Here you use Lucene directly: you index about 45,000 movies, then add keyword
search, a metadata filter on the release year, and fuzzy matching for misspelled queries.

The notebook stays in Python. It prepares the data, drives your Java program, and checks its
output. Your code lives in a small Java file, <a href="lucene/src/main/java/mmir/Search.java">lucene/src/main/java/mmir/Search.java</a>, where you
implement two methods. A Python helper compiles and runs it for you, so you never leave the
notebook.

> **Requires a JDK (Java 21 or newer).** Lucene 10 needs Java 21+. Check with `javac -version` in a
> terminal. If it is missing or older, the setup cell below prints an install hint for your system.
> The Lucene libraries themselves are downloaded automatically the first time you run it.

Run the setup cell first. It makes the `shared/` package importable, then loads the helpers.

In [ ]:
# Standard setup - run this first
import sys, pathlib

# Make `shared/` importable regardless of notebook depth
# (exercise notebooks live in exercises/chNN/, solutions in exercises/chNN/solution/).
_p = pathlib.Path().resolve()
while not (_p / "shared").is_dir() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

%load_ext autoreload
%autoreload 2

from shared.collections import load_collection
from shared.display import print_table, display_md
from shared.lucene_harness import LuceneProject

The data is the movie dataset from the Chapter 1 exercise, this time the **full collection** of about
45,000 movies. Each movie has a title, a combined searchable text (title, overview, tagline, cast,
and genres), and a release year. The first load downloads the dataset, which can take a minute.
Lucene reads its input from a file, so the cell below writes the collection to a tab-separated file
that your Java program will index.

In [ ]:
movies = load_collection("movies")

def _clean(value: str) -> str:
    """Collapse whitespace so each field stays on one tab-separated line."""
    return " ".join(str(value).split())

data_file = pathlib.Path("movies.tsv")
with open(data_file, "w", encoding="utf-8") as f:
    for doc in movies:
        f.write("\t".join([doc["id"], _clean(doc["title"]), _clean(doc["text"]),
                            str(doc.get("year", 0))]) + "\n")

display_md(f"**Wrote {len(movies)} movies** to `{data_file}` (fields: id, title, text, year).")

The next cell creates the project handle and compiles the Java code. It checks that a suitable JDK
is present and downloads the Lucene libraries on first use. It compiles even before you fill anything
in, because the two methods start as stubs; the index and search cells below will fail until you
implement them.

In [ ]:
project = LuceneProject("lucene")   # the lucene/ folder next to this notebook
project.compile()

> **How this works:** Implement the two methods in <a href="lucene/src/main/java/mmir/Search.java">lucene/src/main/java/mmir/Search.java</a>. Write the
> code yourself or direct an AI to implement it from the spec. Save the file, then re-run the cells
> below: the helper recompiles automatically, so you never restart the kernel. You can also open the
> <a href="lucene/">lucene/</a> folder in an IDE (it is a Maven project) for autocompletion while you edit.

### 💻 Task C1 - A Lucene search engine

<a href="lucene/src/main/java/mmir/Search.java">Search.java</a> is a command-line tool with two subcommands: `index` builds the index from the data
file, and `query` runs a search and prints the results as JSON. The plumbing (reading the file,
running the search, printing JSON, parsing arguments) is written for you. You implement the two
methods that carry the retrieval logic:

- **`buildDocument(id, title, text, year)`** turns one record into a Lucene `Document`. Add four
  fields: `id` matched exactly (not tokenized) and stored; `title` tokenized and stored; `text`
  tokenized and searchable; and `year` as a numeric field that supports range queries. The query
  side searches the `text` field and filters on the `year` field.
- **`buildQuery(q, yearMin, fuzzy, analyzer)`** builds the Lucene query. Parse the free-text query
  `q` against the `text` field using the given analyzer, so the query is analyzed the same way the
  documents were. When `fuzzy` is true, each word should match approximately, so a small spelling
  mistake still finds the document. When `yearMin` is not null, restrict the results to documents
  whose `year` is at least `yearMin`, without letting that restriction change the relevance scores.

Both methods have a comment marking where your code goes. Fill them in until every assertion in the
verify cell passes.

First build the index:

In [ ]:
# --- Setup C1 ---
# Runs `Search index`, which calls your buildDocument for every movie.
print(project.index(data_file))   # -> {'indexed': 500} once buildDocument is implemented

In [ ]:
# --- Verify C1 ---
# A relevant query returns ranked hits: at most k of them, sorted by descending score.
res = project.query("space adventure", k=5)
assert res, "expected hits for 'space adventure'"
assert len(res) <= 5, f"should return at most k=5, got {len(res)}"
scores = [h["score"] for h in res]
assert scores == sorted(scores, reverse=True), "results must be sorted by descending score"
assert len(project.query("love", k=5)) == 5, "a common query should fill all k slots"

# A distinctive query surfaces the matching films near the top.
jp = project.query("jurassic park", k=5)
assert any("Jurassic" in h["title"] for h in jp), f"expected a Jurassic film in {[h['title'] for h in jp]}"

# The year filter keeps only recent films and never returns an older one.
assert any(h["year"] < 2000 for h in jp), "unfiltered 'jurassic park' should include a pre-2000 film"
recent = project.query("jurassic park", k=5, year_min=2000)
assert recent, "expected some Jurassic films from 2000 or later"
assert all(h["year"] >= 2000 for h in recent), "every filtered hit must be from 2000 or later"

# Fuzzy matching: a misspelled term finds the films only when fuzzy is on.
assert project.query("jurasic", k=5) == [], "a misspelled term should not match without fuzzy"
assert any("Jurassic" in h["title"] for h in project.query("jurasic", k=5, fuzzy=True)), \
    "fuzzy search should recover the Jurassic films"

# Edge case: an empty query returns nothing.
assert project.query("", k=5) == [], "empty query should return []"

print("✓ Search.java passes every check!")

In [ ]:
# --- Explore C1 ---
# Try your own searches and read the ranked results.
def show(query, k=5, **kwargs):
    display_md(f"**Top {k} for `{query}`** {kwargs if kwargs else ''}:")
    print_table([[f"{h['score']:.3f}", h["year"], h["title"]] for h in project.query(query, k=k, **kwargs)],
                headers=["score", "year", "title"])

show("space adventure with aliens")
show("romance in paris", year_min=2000)
# Which query gives the most surprising top result? Turn fuzzy on and off on a misspelled title:
# how far can the spelling drift before the right movie stops appearing?

### ✏ Task T2 - The same document, two scores

The Lucene index you built runs on one machine. Production engines like Elasticsearch and OpenSearch
(both built on Lucene) split the collection into **shards**, each a separate Lucene index on its own
machine, holding a distinct subset of the documents. Each shard keeps its own term statistics. A
colleague moves one document from one shard to another and notices that its BM25 score changes
slightly, even though the document's text is identical. Recall the BM25 scoring formula from the
chapter:

$$\text{sim}_{\text{BM25}}(Q, D) = \sum_{j} \text{idf}(t_j) \cdot \frac{\text{tf}(D, t_j)\,(k + 1)}{\text{tf}(D, t_j) + k\left(1 - b + b\,\frac{\lvert D \rvert}{\text{avgdl}}\right)}, \quad \text{idf}(t_j) = \log \frac{N - \text{df}(t_j) + 0.5}{\text{df}(t_j) + 0.5}.$$

1. Explain why the score changes. Identify which quantities in the formula a shard computes from its
   own local documents rather than from the whole collection, and trace how moving the document
   changes them.
2. The engines ship this anyway. Explain why these small score differences are acceptable for search,
   and name one situation where they would not be.
3. Sharding lets the collection grow past what one machine holds, but a single query still fans out
   across every shard. Explain what sharding buys for the system as a whole, and what it does *not*
   do for the latency of one query.

> *Your answer here...*